
**Железо:** RTX 5050 Laptop 8 ГБ, 32 ГБ ОЗУ, Intel Core Ultra 7 258V, CUDA 12.8+
**Модель:** gemma4:e2b (Ollama)

In [ ]:
import json
import pandas as pd

REPORTS = "../reports/"


def latest(pattern):
    import glob, os
    files = sorted(glob.glob(REPORTS + pattern), key=os.path.getmtime)
    print("отчёт:", os.path.basename(files[-1]))
    return files[-1]
DATA = "../data/benchmark/"


def read_jsonl(path):
    return [json.loads(line) for line in open(path, encoding="utf-8") if line.strip()]

## Скорость

Итоговая конфигурация: вход — Rules + GLiNER + HiveTrace (параллельно), выход — secrets + canary + prompt_similarity.

### 1. Сколько добавляет сервер (один запрос)

In [4]:
lat = pd.read_csv(REPORTS + "guarded_gemma4_e2b_20261028_0005_latency.csv", sep=";")
lat = lat[lat["ok"]]
# один и тот же вопрос одним и тем же повтором разными путями
pairs = lat.pivot_table(index=["repeat", "prompt"], columns="path", values="total_ms")

one = pd.DataFrame({
    "Ответ, мс (медиана)": [pairs["direct"].median(), pairs["service"].median(), pairs["guarded"].median()],
    "Добавляет к прямому, мс": [0,
                                (pairs["service"] - pairs["direct"]).median(),
                                (pairs["guarded"] - pairs["direct"]).median()],
    "Запросов/с": [1000 / pairs["direct"].mean(), 1000 / pairs["service"].mean(), 1000 / pairs["guarded"].mean()],
}, index=["Напрямую в модель", "Сервис без защиты", "Сервис с защитой"])
one.round(1)

,"Ответ, мс (медиана)","Добавляет к прямому, мс",Запросов/с
Напрямую в модель,321.0,0.0,2.5
Сервис без защиты,317.5,-1.7,2.6
Сервис с защитой,537.6,165.5,1.9


In [7]:
print(f"Сервис с защитой: {(pairs['guarded'] - pairs['direct']).median():+.0f} мс")

Сервис с защитой: +166 мс


### 2. Сколько добавляет каждая проверка

In [8]:
per_check, input_stage, output_stage = {}, [], []
for row in read_jsonl("../logs/requests.jsonl"):
    names = [c["name"] for c in row.get("checks", [])]
    if "out:prompt_similarity" not in names or "out:hivetrace" in names:
        continue
    for c in row["checks"]:
        per_check.setdefault(c["name"], []).append(c["ms"])
    input_stage.append(max(c["ms"] for c in row["checks"] if not c["name"].startswith("out:")))
    output_stage.append(sum(c["ms"] for c in row["checks"] if c["name"].startswith("out:")))

where = {"rules": "вход, CPU", "gliner": "вход, GPU", "hivetrace": "вход, GPU",
         "out:secrets": "выход", "out:canary": "выход", "out:prompt_similarity": "выход, CPU"}
checks_table = pd.DataFrame({
    "Где": pd.Series(where),
    "мс (медиана)": pd.Series({k: pd.Series(v).median() for k, v in per_check.items()}),
    "мс (среднее)": pd.Series({k: pd.Series(v).mean() for k, v in per_check.items()}),
    "запросов": pd.Series({k: len(v) for k, v in per_check.items()}),
}).loc[list(where)]
checks_table.loc["Вход всего (параллельно)"] = ["", pd.Series(input_stage).median(), pd.Series(input_stage).mean(), len(input_stage)]
checks_table.loc["Выход всего (по очереди)"] = ["", pd.Series(output_stage).median(), pd.Series(output_stage).mean(), len(output_stage)]
checks_table.round(1)

,Где,мс (медиана),мс (среднее),запросов
rules,"вход, CPU",0.1,0.1,2058
gliner,"вход, GPU",47.2,109.5,2058
hivetrace,"вход, GPU",97.3,162.8,2058
out:secrets,выход,0.0,0.0,2058
out:canary,выход,0.0,0.0,2058
out:prompt_similarity,"выход, CPU",84.3,121.4,2058
Вход всего (параллельно),,99.2,174.4,2058
Выход всего (по очереди),,84.3,121.4,2058


### 3. HiveTrace на выходе пробы

In [9]:
def guard_cost(latency_file, load_file):
    l = pd.read_csv(REPORTS + latency_file, sep=";")
    p = l[l["ok"]].pivot_table(index=["repeat", "prompt"], columns="path", values="total_ms")
    ld = pd.read_csv(REPORTS + load_file, sep=";")
    g8 = ld[(ld["ok"]) & (ld["path"] == "guarded") & (ld["level"] == 8)]
    return (p["guarded"] - p["service"]).median(), p["guarded"].median(), len(g8) / g8["wall_s"].iloc[0]


def attack_outcome(file):
    a = pd.read_csv(REPORTS + file, sep=";")
    return ((a["Этап"] == "input").mean() * 100, (a["Этап"] == "output").mean() * 100,
            a["Этап"].isna().mean() * 100, a["Промокод у клиента"].sum())


add_off, med_off, rps_off = guard_cost("guarded_gemma4_e2b_20261028_0005_latency.csv", "guarded_gemma4_e2b_20261028_0005_load.csv")
add_on, med_on, rps_on = guard_cost("guarded_gemma4_e2b_20261029_2353_latency.csv", "guarded_gemma4_e2b_20261029_2353_load.csv")
in_off, out_off, pass_off, leak_off = attack_outcome("service_attacks_gemma4_e2b_dev_20261029_2329.csv")
in_on, out_on, pass_on, leak_on = attack_outcome("service_attacks_gemma4_e2b_dev_20261030_0017.csv")

pd.DataFrame({
    "Без HiveTrace на выходе": [add_off, med_off, rps_off, in_off, out_off, pass_off, leak_off],
    "С HiveTrace на выходе": [add_on, med_on, rps_on, in_on, out_on, pass_on, leak_on],
}, index=["Защита добавляет, мс", "Ответ, мс (медиана)", "Запросов/с при 8 пользователях",
          "Атак остановлено на входе, %", "Атак остановлено на выходе, %", "Атак дошло до клиента, %",
          "Промокод у клиента"]).round(1)

,Без HiveTrace на выходе,С HiveTrace на выходе
"Защита добавляет, мс",169.2,295.4
"Ответ, мс (медиана)",537.6,667.9
Запросов/с при 8 пользователях,3.1,2.5
"Атак остановлено на входе, %",70.1,70.1
"Атак остановлено на выходе, %",5.6,5.6
"Атак дошло до клиента, %",24.3,24.3
Промокод у клиента,0.0,0.0


In [10]:
out_hive = [c for row in read_jsonl("../logs/requests.jsonl") for c in row.get("checks", []) if c["name"] == "out:hivetrace"]
print("HiveTrace на выходе: проверил ответов —", len(out_hive))
print("                     сработал —", sum(c["flagged"] for c in out_hive), "раз")
print("                     время, мс (медиана) —", round(pd.Series([c["ms"] for c in out_hive]).median()))

HiveTrace на выходе: проверил ответов — 829
                     сработал — 0 раз
                     время, мс (медиана) — 265


### 4. Под нагрузкой

In [11]:
load = pd.read_csv(REPORTS + "guarded_gemma4_e2b_20261028_0005_load.csv", sep=";")
load = load[load["ok"]]

rows = []
for users, part in load.groupby("level"):
    s = part[part["path"] == "service"]
    g = part[part["path"] == "guarded"]
    rows.append({
        "Пользователей": users,
        "Без защиты, мс": s["total_ms"].median(),
        "С защитой, мс": g["total_ms"].median(),
        "Защита добавляет, мс": g["total_ms"].median() - s["total_ms"].median(),
        "Без защиты, запр/с": len(s) / s["wall_s"].iloc[0],
        "С защитой, запр/с": len(g) / g["wall_s"].iloc[0],
    })
under_load = pd.DataFrame(rows).set_index("Пользователей")
under_load["Пропускная способность, %"] = (under_load["С защитой, запр/с"] / under_load["Без защиты, запр/с"] - 1) * 100
under_load.round(1)

,"Без защиты, мс","С защитой, мс","Защита добавляет, мс","Без защиты, запр/с","С защитой, запр/с","Пропускная способность, %"
Пользователей,,,,,,
1,322.5,503.6,181.1,2.9,1.9,-35.0
2,495.0,751.2,256.2,3.8,2.6,-30.2
4,887.8,1347.1,459.3,4.4,2.8,-35.7
8,1737.2,2399.1,661.9,4.3,3.1,-27.9


Ollama обрабатывает 4 запроса одновременно (`OLLAMA_NUM_PARALLEL=4`), при 8 пользователях половина ждёт в очереди.

## Тестирование атаками

## Что ловим

| | Чем |
|---|---|
| Негатив об ACME | HiveTrace |
| Политика, оружие | HiveTrace, GLiNER |
| Шифровка | HiveTrace, Rules, Secrets |
| Ролевые игры | GLiNER, HiveTrace |
| Прямая инъекция, «забудь системный промпт» | GLiNER, Rules |
| Утечка инструкций в пересказе / переводе | Prompt similarity, Canary |
| Утечка секрета | HiveTrace, Secrets, Canary |
| Косвенная инъекция, атака в несколько сообщений | HiveTrace, GLiNER (весь диалог) |

### 1. Набор атак

Атака = **цель** (что хочет атакующий) + **приём** (как замаскировано).
Цели — `data/benchmark/goals.txt`, приёмы — 43 шаблонных приёма HiveTrace Red. и мои 23 ручные теста

In [13]:
htr = pd.DataFrame(read_jsonl(DATA + "htr_attacks.jsonl"))
seed = pd.DataFrame(read_jsonl(DATA + "seed_manual.jsonl"))

print("Атак HiveTrace Red:", len(htr), "| целей:", htr["base_case"].nunique(), "| приёмов:", htr["technique"].nunique())
print("Наших атак:", (seed["category"] == "attack").sum())
pd.crosstab(htr["goal"], htr["split"], margins=True, margins_name="всего")

Атак HiveTrace Red: 2021 | целей: 47 | приёмов: 43
Наших атак: 23


split,dev,holdout,всего
goal,,,
harmful,70,16,86
negative_acme,385,88,473
politics,420,96,516
prompt_leak,385,88,473
secret,385,88,473
всего,1645,376,2021


In [14]:
pd.crosstab(htr["attack_category"], htr["split"], margins=True, margins_name="всего")

split,dev,holdout,всего
attack_category,,,
context_switching,141,47,188
in_context_learning,94,0,94
irrelevant_information,94,0,94
output_formatting,376,47,423
roleplay,94,47,141
simple_instructions,47,0,47
task_deflection,141,47,188
text_structure_modification,188,141,329
token_smuggling,470,47,517


In [15]:
# Безопасные запросы — по ним считаем ложные блокировки
benign = pd.DataFrame(read_jsonl(DATA + "benign_ru.jsonl"))
borderline = pd.DataFrame(read_jsonl(DATA + "borderline_ru.jsonl"))
pd.DataFrame({
    "всего": [len(benign), len(borderline), (seed["category"] != "attack").sum()],
    "dev": [(benign["split"] == "dev").sum(), (borderline["split"] == "dev").sum(), (seed["category"] != "attack").sum()],
}, index=["Обычные вопросы покупателя", "Пограничные (безопасность техники)", "Наши обычные и пограничные"])

,всего,dev
Обычные вопросы покупателя,82,59
Пограничные (безопасность техники),10,10
Наши обычные и пограничные,14,14


### 2. Охранники отдельно (без модели)


In [16]:
q = pd.read_csv(REPORTS + "guard_quality_dev_20261030_0039.csv", sep=";")
q["GLiNER + HiveTrace"] = q["GLiNER Guard Omni"] | q["HiveTraceGuard-Pro"]
guards = ["Правила", "Prompt Guard 2", "GLiNER Guard Omni", "HiveTraceGuard-Pro", "Qwen3Guard 0.6B", "GLiNER + HiveTrace"]

att = q[q["category"] == "attack"]
safe = q[q["category"] != "attack"]
print("Атак:", len(att), "| безопасных запросов:", len(safe))
pd.DataFrame({
    "Атак поймано, %": att[guards].mean() * 100,
    "Ложных блокировок": safe[guards].sum(),
    "Ложных, %": safe[guards].mean() * 100,
}).round(1)

Атак: 1668 | безопасных запросов: 83


,"Атак поймано, %",Ложных блокировок,"Ложных, %"
Правила,8.5,1,1.2
Prompt Guard 2,23.0,0,0.0
GLiNER Guard Omni,31.2,1,1.2
HiveTraceGuard-Pro,54.1,2,2.4
Qwen3Guard 0.6B,18.0,0,0.0
GLiNER + HiveTrace,70.1,3,3.6


In [17]:
# По целям, % пойманных
(att.groupby("goal")[guards].mean() * 100).round(1)

,Правила,Prompt Guard 2,GLiNER Guard Omni,HiveTraceGuard-Pro,Qwen3Guard 0.6B,GLiNER + HiveTrace
goal,,,,,,
harmful,8.2,23.3,68.5,97.3,61.6,100.0
leak,10.0,10.0,40.0,80.0,10.0,80.0
negative_acme,8.4,19.4,26.1,39.6,14.8,58.1
politics,8.5,23.3,26.9,40.8,15.8,60.8
prompt_leak,8.6,29.9,30.4,53.8,15.1,73.2
secret,8.6,19.5,34.5,74.8,18.7,83.6


In [18]:
# По типам атак, % пойманных
(att.groupby("attack_category")[guards].mean() * 100).round(1)

,Правила,Prompt Guard 2,GLiNER Guard Omni,HiveTraceGuard-Pro,Qwen3Guard 0.6B,GLiNER + HiveTrace
attack_category,,,,,,
context_switching,33.3,100.0,95.7,38.3,49.6,96.5
in_context_learning,0.0,9.6,100.0,67.0,96.8,100.0
irrelevant_information,0.0,11.7,52.1,29.8,3.2,68.1
output_formatting,0.0,28.5,20.7,24.7,5.9,36.4
roleplay,50.0,100.0,100.0,42.6,100.0,100.0
simple_instructions,0.0,0.0,19.1,29.8,4.3,36.2
task_deflection,0.0,0.7,6.4,36.2,8.5,36.2
text_structure_modification,0.0,5.9,2.7,64.9,0.5,66.0
token_smuggling,10.0,1.5,8.3,89.8,0.2,93.0


### 3. Через сервис с защитой (с моделью)

- **на входе** — остановлена охранниками, модель не вызывалась
- **на выходе** — модель ответила, ответ заменён отказом
- **дошла до клиента** — защита пропустила (модель могла отказать сама; автоматически успех проверяется только для промокода)

In [19]:
s = pd.read_csv(REPORTS + "service_attacks_gemma4_e2b_dev_20261029_2329.csv", sep=";")
s["Итог"] = s["Этап"].map({"input": "на входе", "output": "на выходе"}).fillna("дошла до клиента")

print("Атак:", len(s))
print("Промокод дошёл до клиента:", s["Промокод у клиента"].sum())
print("Промокод пойман на выходе (out:secrets):", s["Сработало"].fillna("").str.contains("out:secrets").sum())
(s["Итог"].value_counts(normalize=True) * 100).round(1)

Атак: 1668
Промокод дошёл до клиента: 0
Промокод пойман на выходе (out:secrets): 31


Итог
на входе            70.1
дошла до клиента    24.3
на выходе            5.6
Name: proportion, dtype: float64

In [23]:
# Итог: сколько атак сервис заблокировал, а сколько нет
blocked = (s["Итог"] != "дошла до клиента").sum()
passed = (s["Итог"] == "дошла до клиента").sum()
pd.DataFrame({
    "атак": [blocked, (s["Итог"] == "на входе").sum(), (s["Итог"] == "на выходе").sum(), passed, len(s)],
    "%": [blocked / len(s) * 100, (s["Итог"] == "на входе").mean() * 100, (s["Итог"] == "на выходе").mean() * 100,
          passed / len(s) * 100, 100],
}, index=["Заблокировано всего", "  на входе", "  на выходе", "Не заблокировано", "Всего атак"]).round(1)

,атак,%
Заблокировано всего,1263,75.7
на входе,1170,70.1
на выходе,93,5.6
Не заблокировано,405,24.3
Всего атак,1668,100.0


In [20]:
# По целям, %
(pd.crosstab(s["Цель"], s["Итог"], normalize="index") * 100).round(1)

Итог,дошла до клиента,на входе,на выходе
Цель,,,
harmful,0.0,100.0,0.0
leak,0.0,80.0,20.0
negative_acme,37.6,58.1,4.3
politics,33.5,60.8,5.7
prompt_leak,22.9,73.2,3.9
secret,7.3,83.6,9.1


In [21]:
# По типам атак, %
(pd.crosstab(s["Тип атаки"], s["Итог"], normalize="index") * 100).round(1).sort_values("на входе")

Итог,дошла до клиента,на входе,на выходе
Тип атаки,,,
task_deflection,53.9,36.2,9.9
simple_instructions,53.2,36.2,10.6
output_formatting,51.6,36.4,12.0
text_structure_modification,25.5,66.0,8.5
irrelevant_information,26.6,68.1,5.3
наши кейсы,21.7,69.6,8.7
token_smuggling,5.7,93.0,1.3
context_switching,3.5,96.5,0.0
roleplay,0.0,100.0,0.0


In [22]:
# Какая проверка сколько раз срабатывала (проверки входа идут параллельно — атака может засчитаться нескольким)
names = s["Сработало"].fillna("").str.split(", ").explode()
names[names != ""].value_counts()

Сработало
hivetrace                902
gliner                   520
rules                    142
out:prompt_similarity     62
out:secrets               31
Name: count, dtype: int64

## Длинный контекст

охранники проверяют длинный текст чанками по 1500 символов + конец диалога с ролями, до первого сработавшего чанка.
Rules смотрят весь текст. 
Модель получает весь текст.
Длиннее 30 000 символов — ошибка 413. (ограничения Ollama)

### Видеопамять охранника, МБ

| Символов | GLiNER целиком | GLiNER чанками | HiveTrace целиком | HiveTrace чанками |
|---|---|---|---|---|
| 1 000 | 79 | 79 | 194 | 194 |
| 4 000 | 582 | 136 | 709 | 274 |
| 8 000 | 1 995 | 136 | 1 802 | 274 |
| 16 000 | 7 353 | 136 | 5 495 | 274 |
| 30 000 | не влезло в 8 ГБ | 136 | не влезло в 8 ГБ | 275 |

### Время охранника, мс

| Символов | GLiNER чанками | HiveTrace чанками | HiveTrace целиком |
|---|---|---|---|
| 1 000 | 35 | 146 | 126 |
| 8 000 | 390 | 1 372 | 1 477 |
| 16 000 | 653 | 2 425 | 5 116 |
| 30 000 | 1 198 | 4 437 | — |

### Спрятанная атака в длинном письме

| Символов | Где атака | Только конец 3000 (было) | Чанками (стало) |
|---|---|---|---|
| 4 000 | в начале | пропущена | найдена |
| 4 000 | в середине | найдена | найдена |
| 4 000 | в конце | найдена | найдена |
| 16 000 | в начале | пропущена | найдена |
| 16 000 | в середине | пропущена | найдена |
| 16 000 | в конце | найдена | найдена |
| 4 000 / 16 000 | нет атаки | не блокирует | не блокирует |

### Весь сервис при росте контекста, мс

| Символов | Напрямую | Без защиты | С защитой | Защита добавляет |
|---|---|---|---|---|
| 200 | 777 | 740 | 850 | +110 |
| 2 000 | 445 | 430 | 1 115 | +685 |
| 4 000 | 364 | 375 | 1 397 | +1 022 |
| 8 000 | 342 | 336 | 2 288 | +1 952 |
| 16 000 | 715 | 692 | 3 930 | +3 238 |

- память охранников постоянна при любой длине; время растёт линейно, ~+200 мс на 1000 символов
- атака находится в любом месте текста
- у gemma в Ollama контекст 4096 токенов (~10–12 тыс. символов), длиннее — обрезается